# 24. 다양한 기반 모델 배깅과 보팅 비교 (Bagging & Voting)

동일한 이진 분류 데이터셋에서 KNN, Logistic Regression, Decision Tree 기반 배깅(Bagging) 앙상블과 서로 다른 모델을 결합하는 보팅(VotingClassifier) 성능을 비교합니다.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    'Bagging_KNN': BaggingClassifier(
        estimator=Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))]),
        n_estimators=50, random_state=42
    ),
    'Bagging_LR': BaggingClassifier(
        estimator=Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=500))]),
        n_estimators=50, random_state=42
    ),
    'Bagging_DT': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=50, random_state=42
    ),
}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    print(f'{name:12s} Accuracy: {accuracy_score(y_test, clf.predict(X_test)):.4f}')

In [ ]:
# VotingClassifier (서로 다른 3가지 알고리즘의 결합)
voting_clf = VotingClassifier(
    estimators=[
        ('knn', Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])),
        ('lr', Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=500))])),
        ('dt', DecisionTreeClassifier(random_state=42)),
    ],
    voting='soft'
)
voting_clf.fit(X_train, y_train)
print(f'Voting (soft) Accuracy: {accuracy_score(y_test, voting_clf.predict(X_test)):.4f}')